# DeliveryPulse: контроль качества raw-данных

Цель — воспроизводимо сравнить маленький чистый набор и набор с намеренными дефектами. Вся логика проверки импортируется из `delivery_pulse.quality`; notebook не исправляет raw CSV и не содержит скрытых правил качества.

In [1]:
from datetime import date
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from delivery_pulse.generation import GenerationConfig, generate_dataset
from delivery_pulse.quality import run_quality

workspace = TemporaryDirectory()
root = Path(workspace.name)
root

PosixPath('/tmp/tmpw3ak1bv9')

## 1. Два сопоставимых набора

Одинаковые seed, период и объём изолируют влияние режима дефектов. Все файлы создаются во временном каталоге.

In [2]:
def make_dataset(name: str, defects: bool):
    return generate_dataset(
        GenerationConfig(
            profile="test",
            orders=60,
            seed=42,
            start_date=date(2024, 1, 1),
            months=2,
            output_dir=root / name / "raw",
            inject_quality_issues=defects,
        )
    )


clean = make_dataset("clean", False)
defective = make_dataset("defective", True)
clean_report, clean_profiles, _ = run_quality(clean.output_dir, root / "clean_reports")
defect_report, defect_profiles, _ = run_quality(
    defective.output_dir, root / "defect_reports"
)

## 2. Итог проверки и найденные типы проблем

In [3]:
comparison = pd.DataFrame(
    [
        {
            "dataset": "clean",
            "status": clean_report.status.value,
            "critical": clean_report.critical_count,
            "error": clean_report.error_count,
            "warning": clean_report.warning_count,
        },
        {
            "dataset": "defective",
            "status": defect_report.status.value,
            "critical": defect_report.critical_count,
            "error": defect_report.error_count,
            "warning": defect_report.warning_count,
        },
    ]
)
comparison

,dataset,status,critical,error,warning
0,clean,passed_with_warnings,0,0,2
1,defective,failed,1,7,3


In [4]:
sorted({issue.issue_type for issue in defect_report.issues})

['artificial_overload',
 'broken_foreign_key',
 'business_duplicate',
 'chronology_violation',
 'cost_outlier',
 'duplicate_primary_key',
 'missing_required_value',
 'resource_delivery_overlap',
 'unknown_category']

## 3. Как дефекты искажают будущие метрики

Ниже не выполняется бизнес-анализ: показаны только механические риски. Дубликат меняет знаменатель, пропуск выручки уменьшает полную финансовую выборку, а нарушенная хронология создаёт отрицательную обещанную длительность.

In [5]:
def diagnostic_counts(result):
    orders = result.tables["orders"]
    routes = result.tables["routes"]
    return {
        "orders_rows": len(orders),
        "unique_orders": orders["order_id"].nunique(),
        "route_rows": len(routes),
        "unique_routes": routes["route_id"].nunique(),
        "known_quoted_revenue": int(orders["quoted_revenue"].notna().sum()),
        "invalid_promises": int(
            (orders["promised_delivery_at"] <= orders["requested_pickup_at"]).sum()
        ),
    }


pd.DataFrame(
    {"clean": diagnostic_counts(clean), "defective": diagnostic_counts(defective)}
)

,clean,defective
orders_rows,60,60
unique_orders,60,60
route_rows,6,7
unique_routes,6,6
known_quoted_revenue,60,60
invalid_promises,0,1


## 4. Отдельная оценка полноты детектора по manifest

Только эта секция читает технический manifest. Основной `run_quality` уже завершён и не получает manifest или ожидаемые record ID.

In [6]:
manifest = pd.read_csv(defective.metadata_dir / "quality_issues_manifest.csv")
expected = set(manifest["issue_type"])
detected = {issue.issue_type for issue in defect_report.issues}
coverage = {
    "expected_types": len(expected),
    "detected_expected_types": len(expected & detected),
    "type_coverage": len(expected & detected) / len(expected),
    "missed_types": sorted(expected - detected),
}
coverage

{'expected_types': 7,
 'detected_expected_types': 7,
 'type_coverage': 1.0,
 'missed_types': []}

## 5. Почему raw нельзя исправлять на месте

Изменение raw уничтожило бы воспроизводимость, доказательство исходного дефекта и возможность повторить проверку. Правильный процесс: сохранить raw, объяснить нарушение, при необходимости поместить строку в отдельный карантин, исправить источник и заново сформировать набор.

Ограничения примера: синтетическая выборка мала; статистическая редкость не доказывает ошибку; календаря смен водителей и доступности автомобилей в модели v1 нет, поэтому пересечения рейсов отмечаются предупреждением.

In [7]:
workspace.cleanup()
print("Temporary raw and report directories removed; repository data was not modified.")

Temporary raw and report directories removed; repository data was not modified.
